In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from typing import List
from langchain.schema import Document
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain.embeddings import HuggingFaceEmbeddings
import torch
from pinecone import ServerlessSpec
#from langchain_pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

load_dotenv()
file_path=r"C:\Users\Vidyadhar\Desktop\RAG"

In [3]:
def load_pdf_files(file_path):
    loader = DirectoryLoader(file_path, glob="*.pdf", show_progress=True, loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

def filter_documents_to_min_length(documents: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in documents:
        src=doc.metadata.get("source")
        minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": src}))
    return minimal_docs


def split_documents(documents, chunk_size=500, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    splits = text_splitter.split_documents(documents)
    return splits

In [4]:

def download_embeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": "cuda" if torch.cuda.is_available() else  "cpu"})
    return embeddings
embedings=download_embeddings()

C:\Users\Vidyadhar\AppData\Local\Temp\ipykernel_16828\2999816345.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": "cuda" if torch.cuda.is_available() else  "cpu"})


In [5]:
Documnets=load_pdf_files(file_path)
minimal_docs=filter_documents_to_min_length(Documnets)
splits=split_documents(minimal_docs)

100%|██████████| 5/5 [00:04<00:00,  1.04it/s]


In [6]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY


pine_cone_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pine_cone_api_key)

In [7]:
from pinecone import ServerlessSpec
index_name="test-bot"
if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index=pc.Index(index_name)

In [8]:

from langchain_pinecone import PineconeVectorStore

In [9]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embedings
)

In [11]:
dosearch=vectorstore.add_documents(splits)

In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
result=retriever.invoke(" Iterative Statements")

In [20]:
for doc in result:
    print(doc.page_content)
    print("------------")

Q.  Write a program to check whether the given number is in between 1 and 100? 
1) n=int(input("Enter Number:"))   
2) if n>=1 and n<=10 :   
3)     print("The number",n,"is in between 1 to 10")   
4) else:   
5)     print("The number",n,"is not in between 1 to 10")   
Q. Write a program to take a single digit number from the key board and print is value in 
English word? 
1) 0==>ZERO   
2) 1 ==>ONE   
3)    
4) n=int(input("Enter a digit from o to 9:"))   
5) if n==0 :
------------
Q.  Write a program to check whether the given number is in between 1 and 100? 
1) n=int(input("Enter Number:"))   
2) if n>=1 and n<=10 :   
3)     print("The number",n,"is in between 1 to 10")   
4) else:   
5)     print("The number",n,"is not in between 1 to 10")   
Q. Write a program to take a single digit number from the key board and print is value in 
English word? 
1) 0==>ZERO   
2) 1 ==>ONE   
3)    
4) n=int(input("Enter a digit from o to 9:"))   
5) if n==0 :
------------
Pattern-58: 
1) num=int(